# nb12 -- Pathway / complex / regulatory validation
Checks whether `lasso`'s non-cognate top-predictor genes are known biological
partners of the cognate gene, using three complementary sources:

1. **STRING** -- functional/physical association, broad coverage. Pulls the
   per-evidence-channel scores (experiments, curated databases, co-expression,
   text-mining), not just the combined score, so a hit backed by hard evidence
   can be told apart from one that's mostly literature text-mining.
2. **OmniPath complexes** -- `CORUM, ComplexPortal, hu.MAP`.
3. **TRRUST** -- literature-curated transcription-factor -> target-gene
   regulatory interactions, checked in both directions.

**Model:** `lasso` -- the variant carried forward from nb11 (best cognate
rank-1, 81.6% bootstrap-stable, most reproducible across every stress test run).

**Scope:** the gray-zone proteins -- neither in the trustworthy core (cognate
rank-1 AND bootstrap-stable) nor flagged as likely artifacts (weak raw
correlation / technical gene / large rank gap despite strong raw correlation).
These are the ambiguous cases: the model's top pick isn't the literal cognate
gene, but might still be a real biological partner rather than noise.

## Environment setup

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/covid_citeseq_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## Imports and config

In [ ]:
import sys
sys.path.insert(0, str(BASE_PATH))

import pandas as pd

from src.analysis.pathway_validation import (
    build_trustworthy_core, flag_artifacts, build_query_set,
    fetch_string_partner_cache, string_validate_pairs,
    load_omnipath_complexes, build_gene_to_complexes, complex_validate_pairs,
    load_trrust, trrust_validate_pairs,
    combined_verdict,
    OMNIPATH_COMPLEX_RESOURCES,
)

MODEL_VARIANT = 'lasso'

NB11_RESULTS_DIR = BASE_PATH / 'results' / 'mlp_variant_comparison'
RESULTS_DIR = BASE_PATH / 'results' / 'pathway_validation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_BOOTSTRAP_FREQUENCY = 0.8

## Load nb11 outputs and build the query set
`cognate_ranks`, `bootstrap_rank1`, and `raw_correlation_check` are all
already saved from nb11's run for `lasso` -- no retraining needed here.

In [ ]:
cognate_ranks = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_cognate_ranks.csv')
bootstrap_rank1 = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_bootstrap_rank1.csv')
raw_corr_check = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_raw_correlation_check.csv')

print(f'cognate_ranks:   {len(cognate_ranks)} proteins')
print(f'bootstrap_rank1: {len(bootstrap_rank1)} proteins')
print(f'raw_corr_check:  {len(raw_corr_check)} proteins')

In [ ]:
trustworthy_core = build_trustworthy_core(cognate_ranks, bootstrap_rank1, MIN_BOOTSTRAP_FREQUENCY)
artifact_flags = flag_artifacts(raw_corr_check)
query_set = build_query_set(cognate_ranks, trustworthy_core, artifact_flags)

trustworthy_proteins = set(trustworthy_core.loc[trustworthy_core['trustworthy'], 'protein'])
artifact_proteins = set(artifact_flags.loc[artifact_flags['likely_artifact'], 'protein'])

print(f'Trustworthy core (skipped): {len(trustworthy_proteins)}')
print(f'Artifact-flagged (excluded): {len(artifact_proteins)}')
print(f'Query set: {len(query_set)}')

trustworthy_core.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_trustworthy_core.csv', index=False)
artifact_flags.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_artifact_flags.csv', index=False)
query_set

## 1. STRING check -- combined score AND per-evidence-channel breakdown
`escore` (experimental) and `dscore` (curated databases) are hard-evidence
channels; `tscore` (text-mining) is the weakest, most inflatable one.

In [ ]:
unique_query_genes = pd.unique(query_set[['cognate_gene', 'top_predictor_gene']].values.ravel())
print(f'Fetching STRING partners for {len(unique_query_genes)} unique genes...')

string_partner_cache = fetch_string_partner_cache(unique_query_genes)
print('STRING fetch complete.')

string_results = string_validate_pairs(query_set, string_partner_cache)

print(f"STRING-supported pairs: {string_results['string_known_interaction'].sum()} / {len(string_results)}")
print(f"  ...with hard evidence (experiments or curated database): {string_results['string_hard_evidence'].sum()}")
print(f"  ...text-mining evidence only: {string_results['string_textmining_only'].sum()}")
string_results[string_results['string_known_interaction']].sort_values('string_score', ascending=False)

## 2. OmniPath complexes
`CORUM, ComplexPortal, hu.MAP` -- CORUM alone tends to have narrow curation
scope; hu.MAP adds AP-MS-based complexes rather than literature curation alone.

In [ ]:
complex_cache_path = RESULTS_DIR / f'omnipath_complexes_{OMNIPATH_COMPLEX_RESOURCES.replace(",", "_")}.tsv'
complexes = load_omnipath_complexes(OMNIPATH_COMPLEX_RESOURCES, cache_path=complex_cache_path)
print(f'Complexes loaded ({OMNIPATH_COMPLEX_RESOURCES}): {len(complexes)}')

gene_to_complexes = build_gene_to_complexes(complexes)
print(f'Genes indexed: {len(gene_to_complexes)}')

complex_results = complex_validate_pairs(query_set, gene_to_complexes)
print(f"Complex-supported pairs: {complex_results['shared_complex'].sum()} / {len(complex_results)}")
complex_results[complex_results['shared_complex']]

## 3. TRRUST -- transcription factor -> target regulatory check
Checks whether either gene in a pair is a known TF that regulates the other,
in either direction. If the download fails, get `trrust_rawdata.human.tsv`
manually from https://www.grnpedia.org/trrust/ and place it at
`RESULTS_DIR / 'trrust_rawdata.human.tsv'`.

In [ ]:
trrust_cache_path = RESULTS_DIR / 'trrust_rawdata.human.tsv'
trrust = load_trrust(cache_path=trrust_cache_path)
print(f'TRRUST interactions loaded: {len(trrust)}')

trrust_results = trrust_validate_pairs(query_set, trrust)
print(f"TRRUST-supported (regulatory) pairs: {trrust_results['trrust_regulatory_pair'].sum()} / {len(trrust_results)}")
trrust_results[trrust_results['trrust_regulatory_pair']]

## Combined verdict
A pair counts as biologically supported if STRING, the complex check, or
TRRUST confirms it. `evidence_type` distinguishes physical (complex),
regulatory (TRRUST), and general-association (STRING) support, and separates
hard evidence from text-mining-only STRING hits.

In [ ]:
combined = combined_verdict(string_results, complex_results, trrust_results)
combined = combined.merge(query_set[['protein', 'cognate_rank']], on='protein')

print(f"Total queried: {len(combined)}")
print(f"Biologically supported (any source): {combined['biologically_supported'].sum()}")
print(f"Unexplained (no source supports): {(~combined['biologically_supported']).sum()}")
print('\nBy evidence type:')
print(combined['evidence_type'].value_counts())

combined.sort_values(['biologically_supported', 'string_score'], ascending=[False, False])

## Save results

In [ ]:
combined.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_pathway_validation_results.csv', index=False)
combined[combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_biologically_supported_pairs.csv', index=False)
combined[~combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_pairs.csv', index=False)

print(f'Saved to {RESULTS_DIR}')
print(f'  {MODEL_VARIANT}_trustworthy_core.csv             -- cognate rank-1 AND bootstrap-stable, {trustworthy_core["trustworthy"].sum()}/{len(trustworthy_core)}')
print(f'  {MODEL_VARIANT}_artifact_flags.csv                -- likely-artifact top picks, {artifact_flags["likely_artifact"].sum()}/{len(artifact_flags)}')
print(f'  {MODEL_VARIANT}_pathway_validation_results.csv    -- STRING + complexes + TRRUST, all queried (gray-zone) pairs')
print(f'  {MODEL_VARIANT}_biologically_supported_pairs.csv  -- confirmed by at least one source, with evidence_type')
print(f'  {MODEL_VARIANT}_unexplained_pairs.csv             -- no source supports -- literature spot-check or exclusion candidates')

## Summary
Fill in after running:
- Trustworthy core size (cognate rank-1 AND bootstrap-stable):
- Artifact-flagged size:
- Query set (gray-zone) size:
- Biologically supported fraction of the gray zone:
- Notable regulatory (TRRUST) or complex hits worth highlighting in the pitch:
- Unexplained pairs worth a manual literature check: